In [ ]:
from __future__ import annotations

import asyncio
from typing import Any

from fastapi import FastAPI, HTTPException, Query
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

app = FastAPI(title="Bet365 League Scraper")


EXTRACTOR_JS = r"""
() => {
  function findFirst(node, predicate) {
    if (!node) return null;
    if (predicate(node)) return node;

    for (const child of node._actualChildren || []) {
      const found = findFirst(child, predicate);
      if (found) return found;
    }

    return null;
  }

  function fractionalToDecimal(frac) {
    if (!frac || typeof frac !== "string" || !frac.includes("/")) return null;
    const [a, b] = frac.split("/").map(Number);
    if (!Number.isFinite(a) || !Number.isFinite(b) || b === 0) return null;
    return +(a / b + 1).toFixed(2);
  }

  function extractLeagueMatchesFromStem(stem) {
    const result = {
      leagueId: stem?.data?.ID ?? null,
      topic: stem?.data?.IT ?? null,
      leagueName: null,
      matches: []
    };

    const ev = findFirst(stem, n => n?.nodeName === "EV");
    if (!ev) return result;

    const marketGroups = ev._actualChildren || [];

    const leagueMeta = marketGroups.find(
      n => n?.nodeName === "MG" && n?.data?.ID === "LMAB"
    );
    if (leagueMeta?.data?.CC) {
      result.leagueName = leagueMeta.data.CC;
    }

    const fullTimeGroup = marketGroups.find(
      n => n?.nodeName === "MG" && n?.data?.ID === "40"
    );
    if (!fullTimeGroup) return result;

    const markets = fullTimeGroup._actualChildren || [];

    const teamsMarket = markets.find(
      m => m?.nodeName === "MA" && m?.data?.NA === " "
    );
    const homeMarket = markets.find(
      m => m?.nodeName === "MA" && m?.data?.NA === "1"
    );
    const drawMarket = markets.find(
      m => m?.nodeName === "MA" && m?.data?.NA === "X"
    );
    const awayMarket = markets.find(
      m => m?.nodeName === "MA" && m?.data?.NA === "2"
    );

    const fixtures = new Map();

    for (const pa of teamsMarket?._actualChildren || []) {
      const fi = pa?.data?.FI;
      if (!fi) continue;

      fixtures.set(fi, {
        fixtureId: fi,
        home: pa?.data?.NA ?? null,
        away: pa?.data?.N2 ?? null,
        oddsFractional: { "1": null, "X": null, "2": null },
        oddsDecimal: { "1": null, "X": null, "2": null }
      });
    }

    function mergeOdds(marketNode, key) {
      for (const pa of marketNode?._actualChildren || []) {
        const fi = pa?.data?.FI;
        if (!fi || !fixtures.has(fi)) continue;

        const frac = pa?.data?.OD ?? null;
        fixtures.get(fi).oddsFractional[key] = frac;
        fixtures.get(fi).oddsDecimal[key] = fractionalToDecimal(frac);
      }
    }

    mergeOdds(homeMarket, "1");
    mergeOdds(drawMarket, "X");
    mergeOdds(awayMarket, "2");

    result.matches = Array.from(fixtures.values());
    return result;
  }

  if (
    typeof NavLib === "undefined" ||
    !NavLib?.WebsiteNavigationManager?.CurrentPageData ||
    typeof DataReactLib === "undefined" ||
    typeof DataReactLib.getStemFromLookup !== "function"
  ) {
    return {
      error: "Bet365 runtime no disponible en la página."
    };
  }

  const topic = NavLib.WebsiteNavigationManager.CurrentPageData;
  const stem = DataReactLib.getStemFromLookup(topic);

  if (!stem) {
    return {
      error: "No se encontró stem para el topic actual.",
      topic
    };
  }

  return extractLeagueMatchesFromStem(stem);
}
"""


In [ ]:
async def scrape_league(url: str) -> dict[str, Any]:
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"],
        )

        context = await browser.new_context(
            viewport={"width": 1440, "height": 900},
            locale="es-ES",
        )
        page = await context.new_page()

        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            # Esperar a que exista el runtime que descubriste
            await page.wait_for_function(
                """
                () =>
                  typeof NavLib !== 'undefined' &&
                  typeof DataReactLib !== 'undefined' &&
                  NavLib?.WebsiteNavigationManager?.CurrentPageData &&
                  typeof DataReactLib.getStemFromLookup === 'function'
                """,
                timeout=60000,
            )

            # A veces la UI tarda un poco en hidratar el árbol
            await page.wait_for_timeout(3000)

            data = await page.evaluate(EXTRACTOR_JS)

            if not isinstance(data, dict):
                raise RuntimeError("El extractor devolvió un formato inesperado.")

            if data.get("error"):
                raise RuntimeError(data["error"])

            return data

        except PlaywrightTimeoutError as exc:
            raise RuntimeError("Timeout cargando la liga.") from exc
        finally:
            await context.close()
            await browser.close()


@app.get("/league")
async def get_league(
    url: str = Query(..., description="URL completa de la liga en bet365")
) -> dict[str, Any]:
    try:
        return await scrape_league(url)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc


@app.get("/health")
async def health() -> dict[str, str]:
    return {"status": "ok"}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)